<a href="https://colab.research.google.com/github/gouravbembi/DAA/blob/main/Assignment9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise ValueError("Image could not be loaded!")

lbp = np.zeros_like(img, dtype=np.uint8)

for i in range(1, img.shape[0] - 1):
    for j in range(1, img.shape[1] - 1):

        center = img[i, j]

        neighbors = [
            img[i-1, j-1],
            img[i-1, j],
            img[i-1, j+1],
            img[i, j+1],
            img[i+1, j+1],
            img[i+1, j],
            img[i+1, j-1],
            img[i, j-1]
        ]

        binary = 0

        for k in range(8):
            if neighbors[k] >= center:
                binary += 1 << (7-k)

        lbp[i, j] = binary

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(lbp, cmap='gray')
plt.title("Local Binary Pattern")
plt.axis("off")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(lbp.ravel(), bins=256, range=(0, 256))
plt.title("LBP Histogram")
plt.xlabel("LBP Value")
plt.ylabel("Frequency")
plt.show()


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from scipy.signal import wiener

uploaded = files.upload()
filename = list(uploaded.keys())[0]

img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise ValueError("Image could not be loaded!")

img = img.astype(np.float32) / 255.0

gaussian_noise = np.random.normal(0, 0.05, img.shape)
gaussian_img = np.clip(img + gaussian_noise, 0, 1)

speckle_noise = np.random.normal(0, 0.15, img.shape)
speckle_img = np.clip(img + img * speckle_noise, 0, 1)

def denoise(image):

    image_uint8 = (image * 255).astype(np.uint8)

    wiener_img = wiener(image, (5, 5))
    wiener_img = np.clip(wiener_img, 0, 1)

    median_img = cv2.medianBlur(image_uint8, 5)

    gaussian_img = cv2.GaussianBlur(
        image_uint8, (5, 5), 1.0
    )

    bilateral_img = cv2.bilateralFilter(
        image_uint8, 5, 75, 75
    )

    return [
        wiener_img,
        median_img,
        gaussian_img,
        bilateral_img
    ]

gaussian_results = denoise(gaussian_img)
speckle_results = denoise(speckle_img)

titles = [
    "Wiener Filter",
    "Median Filter",
    "Gaussian Filter",
    "Bilateral Filter"
]

plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(gaussian_img, cmap='gray')
plt.title("Gaussian Noise")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(speckle_img, cmap='gray')
plt.title("Speckle Noise")
plt.axis("off")

for i in range(4):
    plt.subplot(2, 4, 5 + i)
    plt.imshow(gaussian_results[i], cmap='gray')
    plt.title("Gaussian: " + titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(speckle_img, cmap='gray')
plt.title("Speckle Noise")
plt.axis("off")

for i in range(4):
    plt.subplot(2, 3, 3 + i)
    plt.imshow(speckle_results[i], cmap='gray')
    plt.title("Speckle: " + titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()
